# Examples

In [1]:
import numpy as np
from data import *  
from geometry import *
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings("ignore")

## Example 1: Linear Regression (Low-dimensional)

In [2]:
from Regression import linear

### Data Generating Process

In [3]:
# number of source groups = 3, with 1000 samples each
# sigma: source group 1,3: 0.5; source group 2: 2
# target sample size = 10000
# dimension p = 5
n_list = [1000, 1000, 1000]
N = 10000  # target sample size
data = DataContainerSimu_linear_reg_lowd(n_list=n_list, N=N, p=5)
data.generate_funcs_list(seed=0)
data.generate_data(seed=0)

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target


### Implementation \& Results

![Loss Types](loss_type.png)

In [4]:
reg = linear.ld()
reg.fit(Xlist, Ylist, X0, loss_type='reward')
reg.infer(alpha=0.05)

In [5]:
reg.summary()

Model Summary:
CGDRO Aggregated Weights:

group     |        1        2        3
weight_   |   0.4567   0.3451   0.1982

Coefficient Estimators:

index     |        1        2        3        4        5
coef_     |  -0.0655  -0.0433   0.0032  -0.0018   0.0997

Confidence Intervals:

index     |              1              2              3              4              5
CI        | (-0.1283,-0.0027) (-0.1080,0.0214) (-0.0601,0.0665) (-0.0666,0.0629) (0.0351,0.1643)



In [6]:
# Geometry view: the convex hull of the source coefficients (cloesest point to the origin)
beta_source = reg.beta_list
beta_ch, w_ch = nearest_on_convex_hull(beta_source)
print("Estimated coefficients on convex hull:", beta_ch)
print("Weights:", w_ch)

Estimated coefficients on convex hull: [-0.06592632 -0.0432436   0.00284681 -0.00168026  0.09944676]
Weights: [0.45700851 0.34576112 0.19723037]


In [7]:
reg = linear.ld()
reg.fit(Xlist, Ylist, X0, loss_type='squaredloss')
#reg.infer()

In [8]:
reg.summary()

Model Summary:
CGDRO Aggregated Weights:

group     |        1        2        3
weight_   |   0.0000   1.0000   0.0000

Coefficient Estimators:

index     |        1        2        3        4        5
coef_     |  -0.3487  -0.1735  -0.2884  -0.1579  -0.1389

Confidence Intervals not computed. Please run infer() method.


In [9]:
# Geometry view: the sufficiently large noise group dominates
reg.beta_list[1]

array([-0.34866929, -0.17351915, -0.2884043 , -0.15793023, -0.13894055])

In [10]:
reg = linear.ld()
reg.fit(Xlist, Ylist, X0, loss_type='regret')
#reg.infer()

In [11]:
reg.summary()

Model Summary:
CGDRO Aggregated Weights:

group     |        1        2        3
weight_   |   0.3184   0.4467   0.2350

Coefficient Estimators:

index     |        1        2        3        4        5
coef_     |  -0.1004  -0.0816  -0.0368  -0.0537   0.0602

Confidence Intervals not computed. Please run infer() method.


In [12]:
# Geometry view: the center of the minimum enclosing ball of the source coefficients
beta_source = reg.beta_list
beta_cr, r_cr, w_cr = circumcenter_3vectors(beta_source)
print("Estimated coefficients on center of minimum enclosing ball:", beta_cr)
print("Weights:", w_cr)  

Estimated coefficients on center of minimum enclosing ball: [-0.09880893 -0.08368395 -0.03569643 -0.05683792  0.06023717]
Weights: [0.3107334  0.44558458 0.24368202]


## Example 2:Linear Regression (High-dimensionl)

### Data Generating Process

In [13]:
# two source groups, each with 100 samples, and 100 target samples
n_list = [100, 100]
N = 100

data = DataContainerSimu_linear_reg_highd(n_list=n_list, N=N, p=100)
data.generate_funcs_list(seed=0)
data.generate_data(seed=0)

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target


### Implementation \& Results

In [14]:
reg = linear.hd(verbose=True)
reg.fit(Xlist, Ylist, [1,5,10,98], X0=X0)
reg.infer(M=200, alpha=0.05, alpha_thres=0.01)

## time cost: 6.6s

Argument 'loading_intercept' set to False because intercept is False
start fitting-----
======> Bias Correction for initial estimators....
---> Computing for loading (1/4)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (2/4)...
The projection direction is identified at xi = 0.060097 at step = 4.0
---> Computing for loading (3/4)...
The projection direction is identified at xi = 0.060097 at step = 4.0
---> Computing for loading (4/4)...
The projection direction is identified at xi = 0.060097 at step = 4.0
---> Computing for loading (1/4)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (2/4)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (3/4)...
The projection direction is identified at xi = 0.060097 at step = 4.0
---> Computing for loading (4/4)...
The projection direction is identified at xi = 0.060097 at step = 4.0
======> Bias 

In [15]:
reg.summary()

Model Summary:
CGDRO Aggregated Weights:

group     |        1        2
weight_   |   0.1865   0.8135

Plug-in Estimators:

index     |        1        5       10       98
coef_     |   0.0093   0.0952   0.1050   0.0426

Debiased Estimators:

index     |        1        5       10       98
coef_     |   0.2038   0.0184   0.0448   0.4717

Confidence Intervals:

index     |              1              5             10             98
CI        | (-0.1566,0.5643) (-0.4131,0.4498) (-0.3340,0.4237) (0.1135,0.8300)



### Prediction

In [16]:
reg.predict()

array([ 0.57151319, -0.02928899, -0.06033991,  0.11612113, -0.68601212,
       -0.53553375, -0.21904787,  0.13833815, -0.76199818,  1.1203026 ,
        0.44810204,  0.57437405, -0.36665777, -0.53966998,  0.34693913,
       -0.15112281,  0.1967401 , -0.31707418, -0.81371427, -0.44283385,
       -0.30694343,  0.39382992,  1.04627472,  0.24925858, -0.52190778,
        0.91319591,  0.0698671 ,  0.04021008,  0.82834396,  0.02960527,
        0.01608838, -0.22463264,  0.32442729,  0.04313943, -0.81462286,
       -0.17720282, -0.35866089, -0.54523625, -0.37470143, -0.28446907,
       -0.00845137, -0.72892044,  0.63640322,  1.28434837,  0.00191001,
        0.04363812, -0.72227019, -0.18187351,  0.50318242,  0.28319007,
        0.19396803,  0.1015529 ,  0.56790442,  0.05540394, -1.04799782,
       -0.99332377,  0.3305164 ,  1.33258481, -1.28313494,  0.341081  ,
        0.33124495,  0.20624512, -0.02379628,  0.55200241,  0.39877499,
       -0.51265527, -0.82236677,  0.45091854, -0.19443793,  0.00

## Example 3: Machine Learning Regression

In [2]:
from Regression import ml

### Data Generating Process

In [3]:
# number of source groups = 3, each with 10000 samples, and 100000 target samples
# dimension p = 5
# sigma: source group 1,3: 0.5; source group 2: 3.
data = DataContainerSimu_Nonlinear_reg(n=10000, N=100000)
data.generate_funcs_list(L=3, seed=0)
data.generate_data()

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target

### Implementation \& Prediction

In [5]:
drol = ml(f_learner='mlp', w_learner='linear')
drol.fit(Xlist,Ylist,X0, loss_type='reward')

## time cost: 8.7s

In [ ]:
drol.weight_

array([0.22815885, 0.36544498, 0.40639617])

In [21]:
drol.predict()

array([-0.43465976, -0.59452925,  3.56888153, ..., -0.64894895,
       -1.0991031 , -1.90988944])

In [22]:
# Geometry view: the convex hull of the source coefficients (cloesest point to the origin)
pred_source = drol.pred_full_mat.T
pred_ch, w_ch = nearest_on_convex_hull(pred_source)
print("Predictions on convex hull:", pred_ch)
print("Weights:", w_ch)


Predictions on convex hull: [-0.35322758 -0.62952768  3.50452875 ... -0.6281898  -1.07476109
 -1.93525341]
Weights: [0.22545774 0.3337659  0.44077636]


In [23]:
drol = ml(f_learner='xgb', w_learner='kliep')
drol.fit(Xlist,Ylist,X0, loss_type='squaredloss')

## time cost: 8.7s

In [24]:
drol.weight_

array([0., 1., 0.])

In [25]:
drol.predict()

array([-3.88616228,  1.057217  ,  6.17811775, ..., -1.51038933,
       -2.18638206, -0.82921416])

In [26]:
# Geometry view: the sufficiently large noise group dominates
pred_source = drol.pred_full_mat.T
pred_source[1]

array([-3.88616228,  1.057217  ,  6.17811775, ..., -1.51038933,
       -2.18638206, -0.82921416])

In [27]:
drol = ml(f_learner='xgb', w_learner='kliep')
drol.fit(Xlist,Ylist,X0, loss_type='regret')

## time cost: 8.7s

In [28]:
drol.weight_

array([0.41776297, 0.1890659 , 0.39317113])

In [29]:
drol.predict()

array([ 0.62506513, -0.47494394,  2.3271186 , ..., -0.31581939,
       -0.97205993, -2.22075653])

In [30]:
# Geometry view: the center of the minimum enclosing ball of the source coefficients
pred_source = drol.pred_full_mat.T
pred_cr, r_cr, w_cr = circumcenter_3vectors(pred_source)
print("Predictions on center of minimum enclosing ball:", pred_cr)
print("Weights:", w_cr)

Predictions on center of minimum enclosing ball: [ 0.4513905  -0.41582123  2.47527916 ... -0.36179329 -1.01885452
 -2.16717992]
Weights: [0.40170845 0.22029427 0.37799728]


## Example 4: DRlm - Classification

In [7]:
from Classification import Classification

### Data Generating Process

In [8]:
# two source groups, each with 100 samples, and 1000 target samples
n = 100; p = 5; L = 2; N = 1000; K = 2
data = DataContainerSimu_linear_Cl(n=n, N=N, p=p, L=L, K=K)
data.generate_funcs_list(seed=123)
data.generate_data(seed=123)

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target


### Implementation \& Results

In [14]:
cc = Classification(f_learner='linear', w_learner='linear')
cc.fit(Xlist,Ylist,X0)
cc.infer()

## time cost: 4.8s

In [15]:
cc.summary()

Model Summary:
CGDRO Aggregated Weights:

group     |        1        2
weight_   |   0.7985   0.2015

Coefficient Estimators:

Class 1 coefficients:
index     |        1        2        3        4        5
coef_     |   0.0582  -0.1451  -0.0623  -0.0652   0.3645

Class 2 coefficients:
index     |        1        2        3        4        5
coef_     |   0.3192  -0.3334  -0.1658   0.3339  -0.3211

Confidence Intervals:

Class 1 Confidence Intervals:
index     |              1              2              3              4              5
CIs       | (-1.113,1.287) (-1.996,1.440) (-2.633,1.996) (-1.540,3.026) (-1.153,2.826)

Class 2 Confidence Intervals:
index     |              1              2              3              4              5
CIs       | (-0.866,1.729) (-2.148,1.004) (-2.503,1.806) (-1.280,3.821) (-1.507,1.776)



In [35]:
cc.summary(
    index = [3,5], class_index=2
)

Model Summary:
CGDRO Aggregated Weights:

group     |        1        2
weight_   |   0.7985   0.2015

Coefficient Estimators:

Class 2 coefficients:
index     |        3        5
coef_     |  -0.1658  -0.3211

Confidence Intervals:

Class 2 Confidence Intervals:
index     |              3              5
CIs       | (-2.503,1.806) (-1.507,1.776)



### Prediction

In [36]:
cc.predict_proba()

array([[0.46621458, 0.33948019, 0.19430523],
       [0.33735549, 0.3403033 , 0.32234121],
       [0.49823435, 0.25034083, 0.25142482],
       ...,
       [0.3513536 , 0.46685882, 0.18178758],
       [0.34266945, 0.52183562, 0.13549493],
       [0.27539232, 0.28251022, 0.44209746]])

In [37]:
cc.predict()

array([0, 1, 0, 0, 2, 2, 0, 0, 0, 1, 2, 0, 2, 0, 1, 1, 0, 2, 0, 1, 1, 2,
       0, 2, 0, 1, 2, 2, 1, 1, 1, 1, 0, 1, 1, 0, 1, 2, 2, 2, 0, 0, 1, 1,
       1, 0, 2, 1, 1, 0, 2, 1, 1, 1, 1, 1, 1, 0, 2, 2, 1, 1, 2, 0, 1, 1,
       2, 2, 2, 2, 1, 0, 2, 2, 1, 2, 2, 1, 2, 2, 1, 1, 2, 2, 2, 2, 1, 2,
       2, 2, 0, 0, 2, 2, 2, 1, 2, 2, 2, 2, 1, 1, 0, 1, 1, 1, 2, 1, 1, 1,
       1, 1, 1, 2, 2, 2, 1, 0, 2, 2, 2, 2, 1, 2, 0, 2, 0, 1, 1, 0, 1, 0,
       1, 0, 2, 1, 1, 2, 2, 1, 2, 1, 1, 2, 2, 2, 1, 0, 1, 2, 1, 2, 2, 1,
       2, 1, 1, 0, 1, 1, 2, 2, 1, 1, 1, 1, 1, 2, 0, 2, 1, 1, 2, 1, 2, 1,
       1, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 0, 2, 1, 2, 0, 2, 0, 2, 2, 1, 0,
       0, 1, 2, 0, 2, 2, 2, 1, 1, 2, 2, 2, 1, 2, 0, 2, 0, 1, 0, 2, 2, 2,
       2, 1, 2, 1, 1, 0, 1, 1, 1, 2, 1, 2, 1, 2, 0, 2, 2, 0, 1, 0, 1, 2,
       2, 2, 2, 2, 1, 2, 2, 0, 0, 1, 1, 1, 0, 1, 1, 2, 1, 1, 2, 2, 2, 2,
       1, 2, 1, 2, 1, 1, 2, 1, 0, 1, 0, 2, 0, 0, 1, 2, 2, 2, 1, 0, 1, 2,
       2, 2, 2, 2, 2, 1, 1, 2, 2, 1, 2, 2, 1, 2, 2,